# Operations Reasearch: Solution for quiz 6

## Question 1
The formulation is
\begin{array}{rll}
                \min & \displaystyle S & \\[15pt]
                \mbox{s.t.}  
                    & \displaystyle \sum_{i \in I} x_{ij} = 1 & \forall j \in J \\[15pt]
                    & \displaystyle \sum_{g \in C_j} x_{ig} \leq 1 & \forall i \in I, j \in J\\[15pt]
                    & \displaystyle S \geq \sum_{j \in J} p_j x_{ij} & \forall i \in I \\[15pt]
                    & x_{ij} \in \{0, 1\} &\forall i \in I, j \in J.
\end{array}

In [26]:
from gurobipy import *
import pandas as pd

In [35]:
# Read excel sheets and transform them into lists and matrices
problem1 = pd.read_excel('OR2_06_quiz_data.xlsx', 'Problem 1')
#problem1.drop(columns=problem1.columns[-3:], inplace=True)
#print(problem1.columns)
print(problem1.dtypes)
problem1

Job                  int64
Processing time      int64
Conflicting jobs    object
dtype: object


,Job,Processing time,Conflicting jobs
0,1,7,NaN
1,2,4,"5, 8"
2,3,6,NaN
3,4,9,NaN
4,5,12,"2, 8"
5,6,8,9
6,7,10,10
7,8,11,"2, 5"
8,9,8,6
9,10,7,7


In [36]:
problem1['Conflicting jobs'] = problem1['Conflicting jobs'].apply(lambda x: list(map(int, x.split(', '))) if not x!="NaN" else [])

machines = list(range(1, 4))
jobs = problem1['Job'] #.tolist()
processing_time = problem1['Processing time']
conflicting_jobs = problem1['Conflicting jobs']

In [37]:
model_1 = Model("q1")    # build a new model
    
# add variables as a list
x = {}
for i in machines:
    for j in jobs:
        x[i, j] = model_1.addVar(lb = 0, vtype = GRB.BINARY, name = "x" + str(i) + str(j))

makespan = model_1.addVar(lb = 0, vtype = GRB.CONTINUOUS, name = "makespan")
             
# setting the objective function 
model_1.setObjective(makespan, GRB.MINIMIZE) 

# add constraints and name them
model_1.addConstrs((quicksum(x[i, j] for i in machines) == 1 for j in jobs))
model_1.addConstrs((quicksum(x[i, g] for g in conflicting_jobs[j-1]) <= 1 for i in machines for j in jobs))  
model_1.addConstrs((makespan >= quicksum(processing_time[j-1]*x[i, j] for j in jobs) for i in machines))

model_1.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-5500U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 63 rows, 46 columns and 93 nonzeros
Model fingerprint: 0x9a683cf0
Variable types: 1 continuous, 45 integer (45 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 86.0000000
Presolve removed 45 rows and 0 columns
Presolve time: 0.00s
Presolved: 18 rows, 46 columns, 93 nonzeros
Variable types: 0 continuous, 46 integer (45 binary)

Root relaxation: objective 4.266667e+01, 24 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0    

In [38]:
print("Result:")

for i in machines:
    print("Jobs on machine " + str(i) + ": ", end="")
    for j in jobs:
        if x[i,j].x == 1:
            print(str(j) + " ", end="")
    print()
 
print("makespan =", model_1.objVal)    # print objective value

Result:
Jobs on machine 1: 4 5 9 14 
Jobs on machine 2: 1 3 6 11 13 
Jobs on machine 3: 2 7 8 10 12 15 
makespan = 43.0


## Question 2
The formulation is
\begin{array}{rll}
                \min & \displaystyle F & \\[15pt]
                \mbox{s.t.}  
                    & \displaystyle \sum_{i \in I} x_{ij} = 1 & \forall j \in I\\[15pt]
                    & \displaystyle x_{ij} <= y_i & \forall i \in I, j \in J\\[15pt]
                    & \displaystyle \sum_{i \in I} y_{i} = m & \forall i \in I \\[15pt]
                    & \displaystyle w_i \geq \sum_{j \in I} x_{ji} d_{ji} & \forall i \in I\\[15pt]
                    & \displaystyle F \geq w_i p_i & \forall i \in I \\[15pt]
                    & y_{i} \in \{0, 1\} &\forall i \in I \\[15pt]
                    & x_{ij} \in \{0, 1\} &\forall i \in I, j \in J.
\end{array}

In [39]:
problem2 = pd.read_excel('OR2_06_quiz_data.xlsx', 'Problem 2', header=[0,1])
problem2.drop(columns=problem2.columns[-11:], inplace=True)
problem2.columns = ['District (from)', 0,1,2,3,4,5,6,7,'Population']

districts = problem2['District (from)']
distance = problem2.iloc[:, 1:9]
population = problem2['Population']
m = 2

In [40]:
model_2 = Model("q2")    # build a new model
    
# add variables as a list
y = {}
w = {}
x = {}
for i in districts:
     y[i] = model_2.addVar(lb = 0, vtype = GRB.BINARY, name = "y" + str(i))
     w[i] = model_2.addVar(lb = 0, vtype = GRB.INTEGER, name = "w" + str(i))
     for j in districts:
         x[i,j] = model_2.addVar(lb = 0, vtype = GRB.BINARY, name = "x" + str(i) + str(j))

firefighting_times = model_2.addVar(lb = 0, vtype = GRB.CONTINUOUS, name = "firefighting_times")
             
# setting the objective function 
model_2.setObjective(firefighting_times, GRB.MINIMIZE) 

# add constraints and name them
model_2.addConstrs((quicksum(x[i,j] for i in districts) == 1 for j in districts))
model_2.addConstrs((x[i,j] <= y[i] for i in districts for j in districts))
model_2.addConstr((quicksum(y[i] for i in districts) <= m))
model_2.addConstrs((w[i] >= quicksum(x[j, i] * distance[j-1][i-1]  for j in districts) for i in districts))
model_2.addConstrs((firefighting_times >= w[i] * population[i-1] for i in districts))

model_2.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-5500U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 89 rows, 81 columns and 280 nonzeros
Model fingerprint: 0x4603837a
Variable types: 1 continuous, 80 integer (72 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]
Found heuristic solution: objective 240.0000000
Presolve removed 22 rows and 22 columns
Presolve time: 0.00s
Presolved: 67 rows, 59 columns, 208 nonzeros
Variable types: 0 continuous, 59 integer (58 binary)

Root relaxation: objective 8.289474e+01, 52 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0

In [41]:
print("Result:")

for i in districts:
    print(y[i].varName, '=', y[i].x)
# head of the result table
print("\tCity1\tCity2\tCity3\tCity4\tCity5\tCity6\tCity7\tCity8")

for j in districts:
    # mark which product is printed now
    print("City" + str(j), "\t", end="")
    for i in districts:
        print(x[i,j].x, "\t", end="")
    print("")    # use for change line

print("z* =", model_2.objVal)    # print objective value

Result:
y1 = 1.0
y2 = -0.0
y3 = -0.0
y4 = -0.0
y5 = -0.0
y6 = 1.0
y7 = -0.0
y8 = -0.0
	City1	City2	City3	City4	City5	City6	City7	City8
City1 	1.0 	-0.0 	-0.0 	-0.0 	0.0 	0.0 	0.0 	0.0 	
City2 	1.0 	-0.0 	-0.0 	-0.0 	-0.0 	-0.0 	0.0 	0.0 	
City3 	-0.0 	-0.0 	-0.0 	-0.0 	-0.0 	1.0 	-0.0 	0.0 	
City4 	1.0 	-0.0 	-0.0 	-0.0 	-0.0 	-0.0 	-0.0 	-0.0 	
City5 	1.0 	-0.0 	-0.0 	-0.0 	-0.0 	0.0 	-0.0 	-0.0 	
City6 	0.0 	0.0 	-0.0 	-0.0 	-0.0 	1.0 	-0.0 	-0.0 	
City7 	0.0 	0.0 	-0.0 	-0.0 	-0.0 	1.0 	-0.0 	-0.0 	
City8 	0.0 	0.0 	0.0 	-0.0 	-0.0 	1.0 	-0.0 	-0.0 	
z* = 135.0


## Question 3

In [42]:
m = 3
model_3 = Model("q3")    # build a new model
    
# add variables as a list
y = {}
w = {}
x = {}
for i in districts:
     y[i] = model_3.addVar(lb = 0, vtype = GRB.BINARY, name = "y" + str(i))
     w[i] = model_3.addVar(lb = 0, vtype = GRB.INTEGER, name = "w" + str(i))
     for j in districts:
         x[i,j] = model_3.addVar(lb = 0, vtype = GRB.BINARY, name = "x" + str(i) + str(j))

firefighting_times = model_3.addVar(lb = 0, vtype = GRB.CONTINUOUS, name = "firefighting_times")
             
# setting the objective function 
model_3.setObjective(firefighting_times, GRB.MINIMIZE) 

# add constraints and name them
model_3.addConstrs((quicksum(x[i,j] for i in districts) == 1 for j in districts))
model_3.addConstrs((x[i,j] <= y[i] for i in districts for j in districts))
model_3.addConstr((quicksum(y[i] for i in districts) <= m))
model_3.addConstrs((w[i] >= quicksum(x[j, i] * distance[j-1][i-1]  for j in districts) for i in districts))
model_3.addConstrs((firefighting_times >= w[i] * population[i-1] for i in districts))

model_3.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-5500U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 89 rows, 81 columns and 280 nonzeros
Model fingerprint: 0xbdb86ca5
Variable types: 1 continuous, 80 integer (72 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+00]
Found heuristic solution: objective 240.0000000
Presolve removed 22 rows and 22 columns
Presolve time: 0.00s
Presolved: 67 rows, 59 columns, 208 nonzeros
Variable types: 0 continuous, 59 integer (58 binary)

Root relaxation: objective 5.137139e+01, 41 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0

In [43]:
print("Result:")

for i in districts:
    print(w[i].varName, '=', w[i].x)
# head of the result table
print("\tCity1\tCity2\tCity3\tCity4\tCity5\tCity6\tCity7\tCity8")

for j in districts:
    # mark which product is printed now
    print("City" + str(j), "\t", end="")
    for i in districts:
            print(x[i,j].x, "\t", end="")
    print("")    # use for change line

print("z* =", model_3.objVal)    # print objective value

Result:
w1 = 0.0
w2 = 3.0
w3 = 0.0
w4 = 2.0
w5 = 2.0
w6 = 2.0
w7 = 2.0
w8 = 0.0
	City1	City2	City3	City4	City5	City6	City7	City8
City1 	1.0 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	
City2 	1.0 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	
City3 	0.0 	0.0 	1.0 	0.0 	0.0 	0.0 	0.0 	0.0 	
City4 	0.0 	0.0 	1.0 	0.0 	0.0 	0.0 	0.0 	0.0 	
City5 	0.0 	0.0 	1.0 	0.0 	0.0 	0.0 	0.0 	0.0 	
City6 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	1.0 	
City7 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	1.0 	
City8 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	0.0 	1.0 	
z* = 100.0


In [44]:
n = len(districts)

# distance = [
#     [0,3,4,1],
#     [3,0,5,8],
#     [4,5,0,1],
#     [1,8,1,0]
# ]
# distance = pd.DataFrame(distance)

# population = [40, 30, 35, 5]
weighted_distance = distance[0]*population

In [45]:
ambulance_districts = []
for i in range(m): # run m times
    best_dis = -1
    best_firefighting_times = 10000000000000
    for j in range(n):  # calculate the weighted-firefighting time for each district
        weighted_distance = distance[j]*population
        
        if max(weighted_distance) < best_firefighting_times and j not in ambulance_districts:
            best_dis = j
            best_firefighting_times = max(weighted_distance)
    ambulance_districts.append(best_dis)
ambulance_districts

# calculate objective value
min_distance = []
for i in range(n):
    min_dist = 100000
    for j in ambulance_districts:
        if distance[i][j] < min_dist:
            min_dist = distance[i][j]
    min_distance.append(min_dist)

obj = max(pd.DataFrame(min_distance)[0]*population)

In [46]:
obj - model_3.objVal

140.0